In [ ]:
from data import om_us

In [ ]:
print('历史数据入库')
om_us.save_archive()

In [ ]:
print('读取历史数据')
archive_df = om_us.read_archive()

In [ ]:
print('获取预测数据')
forecast_df = om_us.get_forecast(archive_df['date'].max())

In [ ]:
print('数据加工')
process_df = om_us.data_process(archive_df, forecast_df)

In [ ]:
print('绘制图像')
cities_df = om_us.read_cities()
charts_df = om_us.read_charts()
forecast_after = archive_df['date'].max()

# 制图
import src.plt_charts as charts
import matplotlib.pyplot as plt
for i in cities_df.index:
    # 城市参数
    country = cities_df.loc[i]['country']
    city = cities_df.loc[i]['city']
    tag = cities_df.loc[i]['tag']
    code = cities_df.loc[i]['code']
    df = process_df[ process_df['city_code'] == code ].copy()
    for j in charts_df.index:
        params = {
            'min_history_year': charts_df.loc[j]['min_history_year'],
            'forecast_after':forecast_after,
            'ylabel': charts_df.loc[j]['y_label'],
            'title': charts_df.loc[j]['title'] + city + ', ' + country
        }
        if charts_df.loc[j]['variable'] == 'degree_day':
            params['xlim'] = (105, 260)
        chart = charts.day_annul_plot(df, charts_df.loc[j]['variable'], **params)
        path = f'./charts/us/{i:02d}_{code}_{j:02d}_{charts_df.loc[j]['variable']}.jpg'
        chart.savefig(path, dpi=300)
        plt.close()
print('.. 绘制完成')

In [ ]:
print('合成大图')
file_lists = []
for i in cities_df.index:
    code = cities_df.loc[i]['code']
    for j in charts_df.index:
        path = f'./charts/us/{i:02d}_{code}_{j:02d}_{charts_df.loc[j]['variable']}.jpg'
        file_lists.append(path)
charts.merge2grid(file_lists, cities_df.shape[0], charts_df.shape[0], './charts/us/merge_us.jpg')

for idx, idt in enumerate( ['01_precip_sum7', '03_temper']):
    file_lists = []
    for i in cities_df.index:
        code = cities_df.loc[i]['code']
        path = f'./charts/us/{i:02d}_{code}_{idt}.jpg'
        file_lists.append(path)
    charts.merge2grid(file_lists, 2, 3, f'./charts/us/merge_{idt}.jpg')
print('.. 合成完成')